In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import re
import os
print("Libraries Imported")

Libraries Imported


In [2]:
#data cleaning nasal airflow, thoracic, spo2
#           30.05.2024 20:59:00,000; 120

def transform_signal_file(filepath):
    rows=[]
    in_data = False
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if line.lower()=="data:":
                in_data=True
                continue
            if not in_data or not line:
                continue
            parts=line.split(";")
            if len(parts)<2:
                continue
            try:
                ts=pd.to_datetime(parts[0].strip(), format="%d.%m.%Y %H:%M:%S,%f")
                val=float(parts[1].strip())
                rows.append((ts, val))
            except:
                continue
    df=pd.DataFrame(rows, columns=["timestamp","value"])
    df.set_index("timestamp", inplace=True)
    return df

In [3]:
#returns a df
spo2_df=transform_signal_file("Data/AP01/spo2.txt")
spo2_df.head()

,value
timestamp,
2024-05-30 20:59:00.000,93.0
2024-05-30 20:59:00.250,94.0
2024-05-30 20:59:00.500,94.0
2024-05-30 20:59:00.750,94.0
2024-05-30 20:59:01.000,94.0


In [4]:
#datacleaning for flow event file
#    30.05.2024 23:48:45,119-23:49:01,408; 16;Hypopnea; N1

def transform_flow_file(filepath): 
    events=[]
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            m = re.match(r"(\d{2}\.\d{2}\.\d{4})\s+(\d{2}:\d{2}:\d{2},\d+)-(\d{2}:\d{2}:\d{2},\d+);\s*(\d+);\s*([^;]+);\s*(\S+)",
                line)
            if not m:
                continue
            date_str, start_t, end_t, dur, label, stage = m.groups()
            try:
                start=pd.to_datetime(f"{date_str} {start_t}", format="%d.%m.%Y %H:%M:%S,%f")
                end=pd.to_datetime(f"{date_str} {end_t}", format="%d.%m.%Y %H:%M:%S,%f")

                if end<start:
                    end+=pd.Timedelta(days=1)
                events.append({
                        "start" : start,
                        "end" : end,
                        "duration": int(dur),
                        "label":label.strip(),
                        "stage": stage.strip()    
                    })
            except:
                continue
        return events

                    

In [8]:
#returned a dictionary
events = transform_flow_file("Data/AP01/flow_events.txt")
events[0:6]

[{'start': Timestamp('2024-05-30 23:48:45.119000'),
  'end': Timestamp('2024-05-30 23:49:01.408000'),
  'duration': 16,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:50:16.578000'),
  'end': Timestamp('2024-05-30 23:50:33.546000'),
  'duration': 17,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:52:13.626000'),
  'end': Timestamp('2024-05-30 23:52:27.268000'),
  'duration': 14,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:52:51.246000'),
  'end': Timestamp('2024-05-30 23:53:02.871000'),
  'duration': 12,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:53:36.906000'),
  'end': Timestamp('2024-05-30 23:53:49.734000'),
  'duration': 13,
  'label': 'Hypopnea',
  'stage': 'N1'},
 {'start': Timestamp('2024-05-30 23:54:41.454000'),
  'end': Timestamp('2024-05-30 23:54:55.605000'),
  'duration': 14,
  'label': 'Hypopnea',
  'stage': 'N1'}]

In [14]:
#sleep profile
#            30.05.2024 20:59:00,000; Wake
def transform_sleep_profile(filepath):
    rows=[]
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            parts=line.split(";")
            if len(parts)<2:
                continue
            try:
                ts=pd.to_datetime(parts[0].strip(), format="%d.%m.%Y %H:%M:%S,%f")
                stage=parts[1].strip()
                rows.append((ts, stage))
            except:
                continue
    df=pd.DataFrame(rows, columns=["timestamp", "stage"])
    df.set_index("timestamp", inplace=True)
    return df

In [15]:
sleep_df = transform_sleep_profile("Data/AP01/sleep_profile.txt")
sleep_df

,stage
timestamp,
2024-05-30 20:59:00,Wake
2024-05-30 20:59:30,Wake
2024-05-30 21:00:00,Wake
2024-05-30 21:00:30,Wake
2024-05-30 21:01:00,Wake
...,...
2024-05-31 04:32:30,Wake
2024-05-31 04:33:00,Wake
2024-05-31 04:33:30,Wake
